# Prediksi Penyakit Jantung dengan Supervised Learning

**Studi Kasus 4: Diagnosis Penyakit Jantung (Heart Disease UCI Dataset)**

Notebook ini membangun model klasifikasi untuk membantu dokter mengenali pasien yang berisiko tinggi terkena penyakit jantung berdasarkan hasil pemeriksaan medis. Pengerjaan dibagi menjadi lima fase sesuai panduan tugas:

1. Fase 1: Clinical & Data Understanding
2. Fase 2: Data Preparation
3. Fase 3: Modeling (SVM, KNN, Logistic Regression)
4. Fase 4: Model Evaluation & Improvement
5. Fase 5: Clinical Insight & Deployment Concept

Notebook ini dijalankan dari folder `03-notebook`. Seluruh library yang dibutuhkan tercatat di `requirements.txt` pada root project.

## Fase 1: Clinical & Data Understanding

### 1.1 Clinical Understanding

**Masalah klinis.** Penyakit jantung koroner sering tidak bergejala jelas pada tahap awal, sehingga banyak pasien baru terdiagnosis ketika kondisinya sudah serius. Rumah sakit membutuhkan sistem pendukung keputusan klinis (*Clinical Decision Support System*) yang dapat menandai pasien berisiko tinggi dari data pemeriksaan rutin, seperti usia, tekanan darah, kolesterol, hasil EKG, dan hasil tes latihan jantung.

**Tujuan prediksi.** Model memprediksi kolom `target`: 1 berarti pasien diduga memiliki penyakit jantung, dan 0 berarti tidak. Hasil prediksi dipakai sebagai skrining awal untuk menentukan pasien mana yang perlu diperiksa lebih lanjut oleh dokter. Model tidak menggantikan diagnosis dokter.

**Metrik kesuksesan dari sudut pandang medis.** Ada dua jenis kesalahan prediksi:

- *False Negative* (FN): pasien yang sebenarnya sakit diprediksi sehat. Pasien ini bisa pulang tanpa penanganan, padahal risikonya fatal.
- *False Positive* (FP): pasien sehat diprediksi sakit. Akibatnya pasien menjalani pemeriksaan tambahan yang sebenarnya tidak perlu. Ini merugikan dari sisi biaya dan waktu, tapi tidak membahayakan nyawa.

Karena FN jauh lebih berbahaya, metrik utama proyek ini adalah **Recall kelas 1** (proporsi pasien sakit yang berhasil terdeteksi). Precision, F1-Score, dan ROC-AUC tetap dipantau supaya model tidak asal menebak semua pasien sebagai "sakit".

### 1.2 Persiapan Library dan Konstanta

Cell berikut memuat library, menetapkan `RANDOM_STATE = 42` agar hasil dapat direplikasi, dan menyiapkan fungsi `simpan_grafik` untuk menyimpan setiap grafik ke folder `05-output/figures`. Grafik yang tersimpan dipakai kembali di laporan dan slide.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
DATA_PATH = Path("../02-dataset/heart_disease.csv")
FIG_DIR = Path("../05-output/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 20)


def simpan_grafik(fig, nama: str) -> Path:
    """Simpan figure ke FIG_DIR/<nama>.png dan kembalikan path-nya."""
    path = FIG_DIR / f"{nama}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path

### 1.3 Memuat Dataset

Dataset yang dipakai adalah Heart Disease UCI versi Kaggle. Variabel `df_raw` menyimpan data asli dan tidak pernah diubah. Semua proses pembersihan dilakukan pada salinannya.

In [2]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Ukuran data: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
df_raw.head()

Ukuran data: 303 baris x 14 kolom


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### 1.4 Kamus Data

| Kolom | Arti medis | Tipe |
|---|---|---|
| `age` | Usia pasien (tahun) | Numerik |
| `sex` | Jenis kelamin (1 = laki-laki, 0 = perempuan) | Biner |
| `cp` | Tipe nyeri dada (0 = typical angina, 1 = atypical angina, 2 = non-anginal pain, 3 = asymptomatic) | Nominal |
| `trestbps` | Tekanan darah saat istirahat (mm Hg) | Numerik |
| `chol` | Kadar kolesterol serum (mg/dl) | Numerik |
| `fbs` | Gula darah puasa > 120 mg/dl (1 = ya, 0 = tidak) | Biner |
| `restecg` | Hasil EKG saat istirahat (0, 1, 2) | Nominal |
| `thalach` | Detak jantung maksimum saat tes latihan | Numerik |
| `exang` | Nyeri dada (angina) yang dipicu olahraga (1 = ya, 0 = tidak) | Biner |
| `oldpeak` | Depresi segmen ST saat latihan dibanding saat istirahat | Numerik |
| `slope` | Kemiringan segmen ST pada puncak latihan (0, 1, 2) | Nominal |
| `ca` | Jumlah pembuluh darah utama yang terlihat lewat fluoroskopi (0 sampai 3) | Ordinal |
| `thal` | Hasil tes thalium (1 = normal, 2 = fixed defect, 3 = reversible defect) | Nominal |
| `target` | Diagnosis (1 = ada penyakit jantung, 0 = tidak ada) | Target |

### 1.5 Inspeksi Awal

`df.info()` menunjukkan tipe data dan jumlah nilai yang tidak kosong pada tiap kolom. `df.describe()` menampilkan statistik ringkas (rata-rata, sebaran, nilai minimum dan maksimum) untuk melihat apakah ada nilai yang janggal.

In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [4]:
df_raw.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.37,9.08,29.0,47.5,55.0,61.0,77.0
sex,303.0,0.68,0.47,0.0,0.0,1.0,1.0,1.0
cp,303.0,0.97,1.03,0.0,0.0,1.0,2.0,3.0
trestbps,303.0,131.62,17.54,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.26,51.83,126.0,211.0,240.0,274.5,564.0
fbs,303.0,0.15,0.36,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.53,0.53,0.0,0.0,1.0,1.0,2.0
thalach,303.0,149.65,22.91,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.33,0.47,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.04,1.16,0.0,0.0,0.8,1.6,6.2


Semua kolom bertipe numerik dan tidak ada sel kosong. Namun, statistik di atas memperlihatkan nilai minimum `thal` = 0 dan nilai maksimum `ca` = 4. Kedua kode ini tidak ada dalam definisi kamus data, sehingga perlu diperiksa pada tahap berikutnya.

### Pengecekan otomatis: data berhasil dimuat
Cell `assert` ini memastikan dataset terbaca lengkap (303 baris, 14 kolom) dan folder grafik sudah tersedia. Kalau ada yang tidak sesuai, Run All akan berhenti di sini.

In [5]:
assert df_raw.shape == (303, 14)
assert list(df_raw.columns)[-1] == "target"
assert FIG_DIR.exists()
print("Pengecekan data awal: OK")

Pengecekan data awal: OK


### 1.6 Pemeriksaan Kualitas Data: Missing Value dan Duplikat

Pembersihan selalu dimulai dari salinan `df_raw`, sehingga cell ini aman dijalankan berulang kali dan hasilnya tetap sama.

In [6]:
df = df_raw.copy()

print("Jumlah sel kosong per kolom:")
print(df.isna().sum().to_string())
print(f"\nJumlah baris duplikat: {df.duplicated().sum()}")
df[df.duplicated(keep=False)]

Jumlah sel kosong per kolom:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0

Jumlah baris duplikat: 1


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
163,38,1,2,138,175,0,1,173,0,0.0,2,4,2,1
164,38,1,2,138,175,0,1,173,0,0.0,2,4,2,1


Tidak ada sel kosong, tetapi ada **1 baris duplikat** (baris 163 dan 164 identik di semua kolom). Pada data medis, dua pasien dengan hasil pemeriksaan yang sama persis di 14 kolom hampir pasti merupakan data yang tercatat dua kali. Baris duplikat dihapus agar pasien tersebut tidak dihitung dua kali, terutama karena satu salinan bisa masuk ke data latih dan salinan lainnya ke data uji.

In [7]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Ukuran data setelah duplikat dihapus: {df.shape}")

Ukuran data setelah duplikat dihapus: (302, 14)


### 1.7 Missing Value Tersembunyi pada `thal` dan `ca`

Pada dataset UCI asli, `thal` hanya bernilai 1, 2, atau 3, dan `ca` hanya bernilai 0 sampai 3. Pada versi Kaggle ini muncul kode `thal = 0` dan `ca = 4`, yang di dataset asli merupakan data yang hilang (tercatat sebagai `?`). Artinya, data ini sebenarnya punya missing value, hanya saja ditulis sebagai angka.

Kode tersebut diubah menjadi `NaN` supaya tidak dianggap sebagai kategori medis yang sah.

In [8]:
print("Sebelum diubah:")
print(f"  thal = 0 : {(df['thal'] == 0).sum()} baris")
print(f"  ca   = 4 : {(df['ca'] == 4).sum()} baris")

df["thal"] = df["thal"].replace(0, np.nan)
df["ca"] = df["ca"].replace(4, np.nan)

print("\nMissing value setelah diubah:")
print(df[["thal", "ca"]].isna().sum().to_string())

Sebelum diubah:
  thal = 0 : 2 baris
  ca   = 4 : 4 baris

Missing value setelah diubah:
thal    2
ca      4


Jumlah baris yang terdampak kecil (2 baris pada `thal` dan 4 baris pada `ca`), sehingga baris tersebut tidak dibuang. Nilai yang hilang akan diisi dengan **modus** (nilai yang paling sering muncul) pada Fase 2. Pengisian sengaja dilakukan di dalam pipeline setelah *train-test split*, supaya modus dihitung hanya dari data latih dan tidak terjadi *data leakage* (informasi data uji bocor ke proses training).

### Pengecekan otomatis: hasil pembersihan data
Memastikan duplikat sudah hilang, kode `thal=0` dan `ca=4` sudah menjadi NaN, dan `df_raw` tetap utuh.

In [9]:
assert df.shape == (302, 14)
assert df.duplicated().sum() == 0
assert df["thal"].isna().sum() == 2
assert df["ca"].isna().sum() == 4  # 1 dari 5 baris ca=4 ikut terhapus sebagai duplikat
assert (df["thal"] == 0).sum() == 0 and (df["ca"] == 4).sum() == 0
assert df["target"].value_counts().to_dict() == {1: 164, 0: 138}
assert len(df_raw) == 303  # df_raw tidak ikut berubah
print("Pengecekan pembersihan data: OK")

Pengecekan pembersihan data: OK
